<a href="https://colab.research.google.com/github/vladigee/2026_Intro_Python/blob/main/live/part-I/1.5-pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.5-pandas.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch", "xarray": "xarray"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# 1.5) Tabular Data with pandas

pandas handles labelled, tabular data: the `Series` (a labelled 1D array) and the `DataFrame` (named columns sharing an index). This subchapter builds both from a small table of planetary facts, then reads a real year of daily observations from a NOAA climate-reference station — a whitespace-separated text file that writes its missing values as numeric codes — and works through date selection, plotting, resampling, rolling windows, grouping, and missing data. The generated code at the end fills the station's soil-moisture gaps with zero before averaging, and reports the soil about 13 % drier than the measurements say, without raising any error.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/pandas_logo.svg" alt="The pandas project logo" width="500">

<em>The pandas logo, designed by Marc Garcia for the project and taken from <a href="https://commons.wikimedia.org/wiki/File:Pandas_logo.svg">Wikimedia Commons</a>; pandas itself is BSD-3-Clause licensed.</em>

**🎯 Learning objectives**

- Build Series and DataFrames, select from them with `.loc` (labels), `.iloc` (positions), and boolean masks, and derive and modify columns.
- Combine tables with `join`, `merge`, and `concat`.
- Produce a quick plot straight from a Series or DataFrame with `.plot()`.
- Read a messy text file robustly: its separator, its missing-value codes, and typed, date-parsed columns.
- Use a datetime index to select dates and date ranges, resample to coarser periods, and compute rolling windows.
- Recognise the same label/position split in xarray, where `.sel` and `.isel` select along named dimensions.
- Aggregate with `groupby`.
- Treat missing values as physical information: detect with `isna`, and choose `ffill` or `interpolate` deliberately.
- Write results to disk with pathlib paths.

## 1.5.1 Introduction

[pandas](https://pandas.pydata.org/) is an open-source library of data structures and data-analysis tools for *tabular* data: data that can go into a table. If you can imagine the data in a spreadsheet, pandas is the tool for the job.

What it offers, following the list on the pandas website:

- a DataFrame object for data manipulation with integrated indexing;
- tools for reading and writing data between memory and many formats: csv and text files, Excel, SQL databases, and HDF5;
- label-based alignment of data in computations, and integrated handling of missing data;
- reshaping and pivoting of data sets;
- label-based slicing, fancy indexing, and subsetting of large data sets;
- columns that can be inserted and deleted;
- a group-by engine for split-apply-combine operations, which aggregate or transform data;
- merging and joining of data sets;
- hierarchical axis indexing, which holds high-dimensional data in a lower-dimensional structure;
- time-series functionality: date-range generation, frequency conversion, moving-window statistics, and date shifting and lagging;
- critical code paths written in Cython or C, so these operations run fast.

This subchapter covers the basic capabilities. pandas is a deep library, and more advanced use means reading its [documentation](https://pandas.pydata.org/docs/). pandas was created by [Wes McKinney](https://wesmckinney.com/), and several of the examples below follow his book *Python for Data Analysis*, whose [code samples](https://github.com/wesm/pydata-book) are on github.

## 1.5.2 pandas Data Structures: Series

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

A `Series` represents a one-dimensional array of data. The main difference between a `Series` and a numpy array is that a `Series` has an index: the labels used to access the data. There are many ways to create a `Series`; here is one, from a list of values and a list of labels. The numbers are from NASA's [Planetary Fact Sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/).

In [ ]:
names = ["Mercury", "Venus", "Earth"]
values_kg = [0.330e24, 4.87e24, 5.97e24]
mass_kg = pd.Series(values_kg, index=names, name="mass_kg")
mass_kg

In [ ]:
# a Series has plotting methods built in
mass_kg.plot(kind="bar", ylabel="mass (kg)")
plt.show()

In [ ]:
# arithmetic and numpy functions apply element by element, and the result keeps the index
print(mass_kg / 5.97e24)        # in Earth masses
print(np.log10(mass_kg))

### Indexing

In [ ]:
# by label, with .loc
mass_kg.loc["Earth"]

In [ ]:
# by integer position, with .iloc
mass_kg.iloc[2]

In [ ]:
# a list of labels returns several rows
mass_kg.loc[["Venus", "Earth"]]

In [ ]:
# a label slice includes BOTH of its ends
mass_kg.loc["Mercury":"Earth"]

In [ ]:
# a position slice excludes its end, like a list slice
mass_kg.iloc[:2]

The data and the labels can always be taken back out: `.values` returns the values as a numpy array, and `.index` returns the labels as a pandas `Index`.

In [ ]:
print(mass_kg.values)
print(mass_kg.index)

## 1.5.3 pandas Data Structures: DataFrame

A Series holds a single column. A `DataFrame` is a collection of Series that share one index — much like a table in a spreadsheet — and each column has its own dtype. Below, one is built from a dictionary of columns. NASA gives no rotation period for Venus in this table, so that entry is left missing as `np.nan`.

In [ ]:
planets = pd.DataFrame(
    {"mass_kg": [0.330e24, 4.87e24, 5.97e24],
     "diameter_m": [4879e3, 12_104e3, 12_756e3],
     "rotation_period_h": [1407.6, np.nan, 23.9]},
    index=["Mercury", "Venus", "Earth"],
)
planets

pandas keeps track of missing values, shown as `NaN`, through every calculation. `.info()` lists each column with its dtype and the number of values that are not missing.

In [ ]:
planets.info()

Statistical functions work on both Series and DataFrames, one column at a time, and skip missing values by default. `describe` collects the common ones in a single table.

In [ ]:
print(planets.min())
print(planets.mean())
print(planets.std())
planets.describe()

In [ ]:
# one column, as a Series, with square brackets...
print(planets["mass_kg"])
# ...or with attribute syntax, which only works when the column name is a valid Python name
print(planets.mass_kg)

In [ ]:
# rows are selected as in a Series, by label or by position
print(planets.loc["Earth"])
print(planets.iloc[2])

In [ ]:
# a row label and a column label together select one cell
planets.loc["Earth", "mass_kg"]

In [ ]:
# rows 0 and 1 of column 0, by position
planets.iloc[0:2, 0]

A calculation on columns keeps the index, so the result lines up row by row:

In [ ]:
volume_m3 = 4 / 3 * np.pi * (planets["diameter_m"] / 2) ** 3
planets["mass_kg"] / volume_m3

and it can be stored as a new column:

In [ ]:
planets["density_kgm3"] = planets["mass_kg"] / volume_m3
planets

## 1.5.4 Merging Data

pandas has many ways to combine data sets, described in its [merging guide](https://pandas.pydata.org/docs/user_guide/merging.html). `join` combines two tables on their index. Here is a Series of mean surface temperatures that also covers Mars:

In [ ]:
temperature_celsius = pd.Series([167, 464, 15, -65],
                                index=["Mercury", "Venus", "Earth", "Mars"],
                                name="temperature_celsius")
temperature_celsius

In [ ]:
# join returns a new DataFrame; by default it keeps the rows of the left table
planets.join(temperature_celsius)

In [ ]:
# how="right" keeps the rows of the right table instead: Mars appears, with NaN in every
# column that planets has no value for
planets.join(temperature_celsius, how="right")

In [ ]:
# reindex conforms a table to a new list of labels, adding a row of NaN for each label it lacks
everyone = planets.reindex(["Mercury", "Venus", "Earth", "Mars"])
everyone

`merge` joins two tables on a shared *column* rather than on the index, the way a database join does. `concat` stacks tables along an axis; a cell that exists in only one of them comes out as `NaN`.

In [ ]:
orbits = pd.DataFrame({"name": ["Mercury", "Venus", "Earth", "Mars"],
                       "orbital_period_days": [88.0, 224.7, 365.2, 687.0]})
surface = pd.DataFrame({"name": ["Mercury", "Venus", "Earth"],
                        "temperature_celsius": [167, 464, 15]})
merged = surface.merge(orbits, on="name", how="left")
print(merged)

# Mars arrives without an orbital period, so concat leaves that cell NaN
mars = pd.DataFrame({"name": ["Mars"], "temperature_celsius": [-65]})
print(pd.concat([merged, mars], ignore_index=True))

A boolean Series can also index a DataFrame, keeping only the rows where it is `True`. Combine several conditions with `&` (and) and `|` (or) — not Python's `and`/`or` from subchapter 1.2, which do not work element-wise on a Series — and wrap each condition in its own parentheses, since `&`/`|` bind more tightly than comparisons like `==` and `>`, so leaving the parentheses out changes what gets evaluated.

In [ ]:
massive = planets[planets["mass_kg"] > 4e24]
massive

In [ ]:
# two conditions: massive, and rotating once in under 100 hours
planets[(planets["mass_kg"] > 4e24) & (planets["rotation_period_h"] < 100)]

In [ ]:
planets["is_massive"] = planets["mass_kg"] > 4e24
planets

## 1.5.5 Modifying Values

To change values in a DataFrame, select them with `.loc` or `.iloc` and assign. A single label pair changes one cell; a boolean mask inside `.loc` changes only the rows that match it and leaves the rest untouched.

In [ ]:
planets.loc["Earth", "mass_kg"] = 5.972e24       # one cell, by label
planets.loc["Venus", "diameter_m"] += 1          # one cell, updated from its own value
planets.loc[planets["diameter_m"] > 1e7, "size_class"] = "earth-sized"   # matching rows only
planets

Mercury's `size_class` is `NaN`: the masked assignment created the column, wrote a value into the matching rows, and left every other row missing.

## 1.5.6 Plotting

Series and DataFrames carry a `.plot()` method that wraps matplotlib: it reads the index or the named columns for the axes, so a first look needs no figure boilerplate. `kind=` chooses the plot type; the options are listed in the [visualization guide](https://pandas.pydata.org/docs/user_guide/visualization.html).

In [ ]:
planets.plot(kind="scatter", x="mass_kg", y="diameter_m", grid=True)
plt.show()

In [ ]:
planets["density_kgm3"].plot(kind="bar", ylabel="density (kg m$^{-3}$)")
plt.show()

## 1.5.7 Time Indexes

Indexes are a large part of what makes pandas useful, and there are different kinds for different data. For time series, the index is a `DatetimeIndex`. `pd.date_range` builds a run of regularly spaced dates — here, two years of days carrying an idealised annual cycle.

In [ ]:
two_years = pd.date_range(start="2014-01-01", end="2016-01-01", freq="D")
annual_cycle = pd.Series(np.sin(2 * np.pi * two_years.dayofyear / 365), index=two_years)
annual_cycle.plot()
plt.show()

Python's slicing notation inside `.loc` selects a date range by its labels:

In [ ]:
annual_cycle.loc["2015-01-01":"2015-07-01"].plot()
plt.show()

The index also exposes each date's components, such as the month and the day of the month:

In [ ]:
print(annual_cycle.index.month)
print(annual_cycle.index.day)

<details>
<summary><b>🔍 Going deeper: time zones</b></summary>

A naive timestamp has no zone; localize it, then convert.

```python
idx = pd.date_range("2024-06-01", periods=3, freq="h")
aware = idx.tz_localize("UTC")          # attach UTC
local = aware.tz_convert("Europe/Zurich")   # convert to local clock time
```

Store and compute in UTC; convert to local time only for display.

</details>

## 1.5.8 Reading Data Files: Weather Station Data

Real data rarely arrives as a clean comma-separated file. This example uses one year of daily observations from the [U.S. Climate Reference Network](https://www.ncei.noaa.gov/products/land-based-station/us-climate-reference-network) (USCRN), one of NOAA's [land-based station networks](https://www.ncei.noaa.gov/products/land-based-station): station Millbrook 3 W, New York (41.79° N, 73.74° W), for 2017. USCRN files separate their columns with varying amounts of whitespace and write missing values as numeric codes, both documented in the network's [README](https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/README.txt). The copy used here is from a [Zenodo mirror](https://doi.org/10.5281/zenodo.5564850) of the NOAA files (CC-BY-4.0), with a header line of column names added.

The file has 28 columns. The ones used below:

| Column | Quantity | Unit |
|---|---|---|
| `WBANNO` | station identifier | — |
| `LST_DATE` | date, local standard time | YYYYMMDD |
| `T_DAILY_MAX`, `T_DAILY_MIN`, `T_DAILY_MEAN` | daily maximum, minimum, and mean air temperature | °C |
| `P_DAILY_CALC` | daily precipitation | mm |
| `SOLARAD_DAILY` | daily solar radiation | MJ m⁻² |
| `RH_DAILY_AVG` | mean relative humidity | % |
| `SOIL_MOISTURE_5_DAILY` | volumetric soil moisture at 5 cm depth | m³ m⁻³ |
| `SOIL_TEMP_5_DAILY` | soil temperature at 5 cm depth | °C |

A missing value is written −9999.0, except in the soil-moisture columns, whose format has three decimal places and writes it −99.000.

In [ ]:
# Pre-supplied: download the data file and cache it locally.
# You do not need to understand this cell yet — fetching data is covered in the
# reproducible-data-pipelines bonus subchapter.
import pooch
from pathlib import Path

datafile = Path(pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/uscrn_daily_2017_ny_millbrook_3w.txt",
    known_hash="sha256:f97637cf9909548a10843e9af352470b8aa1f9df5d971aa74e7478d63c7eca07",
    fname="uscrn_daily_2017_ny_millbrook_3w.txt",
    path=pooch.os_cache("mlees"),
))

Before reading a file, look at it. Its first lines show the separator and the column names:

In [ ]:
with open(datafile) as f:
    for _ in range(4):
        print(f.readline(), end="")

pandas reads text files with `read_csv`. It can extract data from almost any text file, but only once its options are set to match that file. With no options, this is what we get:

In [ ]:
station = pd.read_csv(datafile)
station.head()

pandas failed to find the columns. It expected a standard [csv (comma-separated values)](https://en.wikipedia.org/wiki/Comma-separated_values) file, found no commas, and put each whole line into one column. In this file the values are separated by whitespace — and not by a single space, since the amount of whitespace varies from column to column. The `sep` keyword says so; `r"\s+"` is a regular expression meaning "one or more whitespace characters".

In [ ]:
station = pd.read_csv(datafile, sep=r"\s+")
station.head()

That worked. Look closely, though: on the very first rows the soil-moisture columns read −99.000, and further down there are values of −9999.0. These are the missing-value codes from the README. Read as numbers, they would enter every statistic as genuine readings — a single −9999 °C day pulls a year's mean temperature down by more than 27 °C. `na_values` tells pandas to read them as missing instead:

In [ ]:
station = pd.read_csv(datafile, sep=r"\s+", na_values=[-9999.0, -99.0])
station.head()

The missing values are now `NaN`. Which data types did pandas infer?

In [ ]:
station.info()

`LST_DATE` came in as an integer, not as a date, and so did the station identifier, although nobody does arithmetic on an identifier. Be explicit: `parse_dates` converts a date column to `datetime64`, and `dtype` pins the type of a key column. Then make the date the index.

In [ ]:
station = pd.read_csv(
    datafile,
    sep=r"\s+",
    na_values=[-9999.0, -99.0],
    parse_dates=["LST_DATE"],       # -> datetime64, not integers
    date_format="%Y%m%d",           # the dates are written as YYYYMMDD
    dtype={"WBANNO": "string"},     # an identifier, not a number
)
station = station.set_index("LST_DATE")
station.info()

With a datetime index, `.loc` selects one day by its date, or a range of days by a slice of dates:

In [ ]:
station.loc["2017-08-07"]

In [ ]:
temperature_columns = ["T_DAILY_MAX", "T_DAILY_MIN", "T_DAILY_MEAN"]
station.loc["2017-07-01":"2017-07-11", temperature_columns]

### Refresher: `.isel` and `.sel` in xarray

xarray (1.4.14) makes the same split between position and label for arrays with any number of dimensions, with one difference: each dimension is selected by its name, not by its place in the array. `.isel` selects by integer position, like `.iloc`; `.sel` selects by coordinate label, like `.loc`, and with `method="nearest"` it also accepts a label that falls between two coordinates. The cell below puts the station's three temperature columns into a 2D `DataArray` with dimensions `time` and `statistic`.

In [ ]:
import xarray as xr

In [ ]:
temp_da = xr.DataArray(
    station[["T_DAILY_MIN", "T_DAILY_MEAN", "T_DAILY_MAX"]].values,
    dims=("time", "statistic"),
    coords={"time": station.index.values, "statistic": ["min", "mean", "max"]},
    attrs={"units": "degC"},
)
print(temp_da.dims, temp_da.shape)

In [ ]:
# .isel: by integer position along each named dimension, like .iloc
print(temp_da.isel(time=0, statistic=1).item())     # mean temperature, first day
print(temp_da.isel(time=-1).values)                 # all three statistics, last day

In [ ]:
# .sel: by coordinate label, like .loc
print(temp_da.sel(statistic="max", time="2017-08-07").item())

In [ ]:
# a label between two days: method="nearest" picks the closer one, 8 August
print(temp_da.sel(time="2017-08-07 18:00", method="nearest")["time"].values)

## 1.5.9 Quick Statistics

In [ ]:
station.describe()

The `count` row is the first thing to read. Most columns hold 364 values, not 365: the station reported nothing at all on 4 October. Soil moisture at 5 cm holds only 317. Those days are now missing, rather than entering the statistics as −9999 or −99.

## 1.5.10 Plotting Values

pandas integrates a [`boxplot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.boxplot.html) method, which draws each column's median, quartiles, and outliers. Columns in different units belong on different axes:

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(10, 8))
station[temperature_columns].boxplot(ax=ax[0, 0])
station[["P_DAILY_CALC"]].boxplot(ax=ax[0, 1])
station[["SOLARAD_DAILY"]].boxplot(ax=ax[1, 0])
station[["RH_DAILY_AVG"]].boxplot(ax=ax[1, 1])
ax[0, 0].set_ylabel("temperature (°C)")
ax[0, 1].set_ylabel("precipitation (mm)")
ax[1, 0].set_ylabel("solar radiation (MJ m$^{-2}$)")
ax[1, 1].set_ylabel("relative humidity (%)")
plt.tight_layout()
plt.show()

In [ ]:
# pandas is time-aware: a datetime index becomes a date axis
station["T_DAILY_MEAN"].plot()
plt.show()

`ax=` draws into axes you created yourself, so a pandas plot can be finished with the matplotlib calls from subchapter 1.4 whenever it needs more control than `.plot()` gives:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
station["T_DAILY_MEAN"].plot(ax=ax)
ax.set_ylabel("temperature (°C)")
ax.set_title("daily mean air temperature")
plt.show()

In [ ]:
station[temperature_columns].plot(figsize=(8, 3), ylabel="temperature (°C)")
plt.show()

Daily precipitation is right-skewed: most days are dry or nearly so, and a few carry most of the year's rain. A linear count axis buries those rare large values near zero; passing `logy=True` puts the count axis on a log scale, so the common and the rare values are visible in the same plot.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
station["P_DAILY_CALC"].plot(ax=ax[0], kind="hist", bins=20, color="tab:gray")
ax[0].set_title("linear count axis")
station["P_DAILY_CALC"].plot(ax=ax[1], kind="hist", bins=20, color="tab:gray", logy=True)
ax[1].set_title("logy=True")
for axis in ax:
    axis.set_xlabel("daily precipitation (mm)")
    axis.set_ylabel("count")
plt.tight_layout()
plt.show()

## 1.5.11 Resampling

Since pandas understands time, it can resample: re-bin a time series to a coarser period. Each period has a frequency string, such as `"MS"` for month start; the full list is in the [time-series documentation](https://pandas.pydata.org/docs/user_guide/timeseries.html#dateoffset-objects).

In [ ]:
# a monthly resampler object: nothing has been computed yet
monthly = station.resample("MS")
monthly

In [ ]:
# numeric_only=True skips the station identifier, which has no mean
monthly.mean(numeric_only=True)

In [ ]:
# the same thing chained into one line, then plotted
station_monthly = station.resample("MS").mean(numeric_only=True)
station_monthly[temperature_columns].plot(ylabel="temperature (°C)")
plt.show()

### Rolling windows

`rolling` computes a moving window instead: each value is replaced by a statistic of the `window` values ending at it. The first `window − 1` results are `NaN` while the window fills, and so is every window that contains a missing day.

In [ ]:
weekly_temp = station["T_DAILY_MEAN"].rolling(window=7).mean()

fig, ax = plt.subplots(figsize=(8, 3))
station["T_DAILY_MEAN"].plot(ax=ax, alpha=0.4, label="daily")
weekly_temp.plot(ax=ax, label="7-day rolling mean")
ax.set_ylabel("temperature (°C)")
ax.legend()
plt.show()

## 1.5.12 groupby: Split, Apply, Combine

`groupby` splits rows by a key, applies an aggregation to each group, and combines the results. The key can be a column, or any array of the same length — here, the month of each date.

In [ ]:
summary = station.groupby(station.index.month).agg(
    mean_temp=("T_DAILY_MEAN", "mean"),      # mean skips NaN
    total_precip=("P_DAILY_CALC", "sum"),
    n_days=("T_DAILY_MEAN", "size"),         # size counts every row, NaN included
)
summary.index.name = "month"
summary.round(2)

`nunique` counts how many distinct values a column holds; `idxmax` (or `idxmin`) returns the *label* of the row holding the maximum (minimum) value — useful directly on a groupby result like `summary` above.

In [ ]:
print("distinct station identifiers:", station["WBANNO"].nunique())
print("warmest month:", summary["mean_temp"].idxmax())
print("wettest month:", summary["total_precip"].idxmax())

`kind="bar"` turns the grouped summary into a labelled bar chart in one line:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
summary["total_precip"].plot(ax=ax, kind="bar", color="tab:blue")
ax.set_ylabel("total precipitation (mm)")
plt.show()

<details>
<summary><b>🔍 Going deeper: multi-index</b></summary>

A hierarchical index lets one frame hold several grouping levels.

```python
wet_day = station["P_DAILY_CALC"] > 0
by_month_and_wet = station.groupby([station.index.month, wet_day])["T_DAILY_MEAN"].mean()
print(by_month_and_wet.loc[7])      # select the outer level: July, dry and wet days
```

`groupby` produces a multi-index like this one whenever you group by more than one key.

</details>

<details>
<summary><b>🔍 Going deeper: apply</b></summary>

`apply` runs an arbitrary function per group or per row when no built-in aggregation fits.

```python
spread = station.groupby(station.index.month)["T_DAILY_MEAN"].apply(lambda s: s.max() - s.min())
```

Prefer vectorised built-ins (`mean`, `sum`, `agg`) where they exist; `apply` is slower and should be the fallback, not the default.

</details>

## 1.5.13 Missing Data as Physical Information

When a sensor stops reporting, the record should show that gap, not invent a reading. pandas marks missing values as `NaN`, detects them with `isna`, and its reductions skip them by default. How you *fill* a gap is a modelling choice: `ffill` carries the last value forward (sensible for slowly varying state), `interpolate` draws a straight line between the neighbours on either side.

In [ ]:
# missing values per column, for the columns that have any
missing = station.isna().sum()
print(missing[missing > 0])

# two fills with different physical meaning, across the one-day gap on 4 October
early_october = station.loc["2017-10-01":"2017-10-07", "T_DAILY_MEAN"]
print("with gaps:  ", early_october.tolist())
print("ffill:      ", early_october.ffill().tolist())                  # carry forward
print("interpolate:", early_october.interpolate().round(2).tolist())  # linear

A gap also carries information in *when* it happens. Most columns miss the single day of 4 October, but soil moisture at 5 cm is missing on 48 days, and 43 of them fall in January and February — on days when the soil temperature at 5 cm sat at or just below 0 °C:

In [ ]:
soil_gap = station["SOIL_MOISTURE_5_DAILY"].isna()
print("soil-moisture gaps per month:", soil_gap.groupby(station.index.month).sum().to_dict())
print("median 5 cm soil temperature on gap days:", station.loc[soil_gap, "SOIL_TEMP_5_DAILY"].median(), "°C")

**🧠 Computational-thinking fundamental: missing is not zero**

A missing value means "we do not know", which is different from any measured number — and very different from zero, a real, often common, physical reading. Encode absence as `NaN` so that reductions skip it and gaps stay visible. Filling is a decision with consequences for every statistic computed afterwards: carry-forward, interpolation, and leaving the gap each suit different physics. A gap filled with 0 °C in a summer temperature record drags every mean and trend computed from it toward freezing, and nothing raises an error.

**ℹ️ Quick exercise: the wettest month by rolling total**

Compute the 30-day rolling total of `P_DAILY_CALC` with `.rolling(window=30).sum()`, then print the date on which it peaks and the peak value, rounded to one decimal.


<details>
<summary><b>✅ Solution</b></summary>

```python
rolling_precip_mm = station["P_DAILY_CALC"].rolling(window=30).sum()
print(rolling_precip_mm.idxmax().date(), round(rolling_precip_mm.max(), 1))
```

</details>

## 1.5.14 Writing Outputs

Save results to disk with a `pathlib` path, which keeps the code OS-independent.

In [ ]:
Path("_files").mkdir(exist_ok=True)
out_path = Path("_files/station_monthly_summary.csv")
summary.to_csv(out_path)
print("wrote", out_path.name, "- exists:", out_path.exists())

<details>
<summary><b>🔍 Going deeper: Polars as a fast alternative</b></summary>

[polars](https://pola.rs/) is a newer dataframe library with a lazy, multi-threaded engine that is often much faster on large data, with a more explicit expression API.

```python
import polars as pl
summary_pl = pl.read_csv("_files/station_monthly_summary.csv")
summary_pl.filter(pl.col("total_precip") > 100).select("month", "mean_temp")
```

The concepts transfer directly from pandas; the syntax differs. It is shown for reference and not run here.

</details>

## *When generated code lies: filling gaps with zero*

Asked to "clean and average" the soil-moisture column, an assistant fills the missing values with zero and then takes the mean. The code runs, but zero is a valid soil moisture — completely dry soil — so each gap becomes a spurious bone-dry day that drags the mean down.

In [ ]:
def mean_soil_moisture(df):
    # fill missing values, then average (as an assistant returned it)
    clean = df["SOIL_MOISTURE_5_DAILY"].fillna(0)
    return clean.mean()

print("buggy mean:  ", round(mean_soil_moisture(station), 3), "m3 m-3")
print("correct mean:", round(station["SOIL_MOISTURE_5_DAILY"].mean(), 3), "m3 m-3")   # skips NaN
print("gaps filled with 0:", int(station["SOIL_MOISTURE_5_DAILY"].isna().sum()))

**⚠️ Diagnosis: fillna(0) invents a measurement**

`fillna(0)` replaced unknown soil moisture with 0 m³ m⁻³, a reading inside the physical range, so the average is biased toward zero — here from 0.189 to 0.165 m³ m⁻³, about 13 % drier, with no error raised. Nearly all the gaps fall in January and February, when the measured days average about 0.23 m³ m⁻³, so each invented 0 replaces a reading from one of the wettest stretches of the year. pandas reductions already skip `NaN`, so the fill was not only wrong but unnecessary. If a gap-free series is genuinely required, `interpolate()` fills each gap from the readings on either side of it, which a constant 0 does not.

In [ ]:
def mean_soil_moisture(df):
    # reductions skip NaN already; interpolate only if a gap-free series is needed
    return df["SOIL_MOISTURE_5_DAILY"].mean()

print("fixed mean:", round(mean_soil_moisture(station), 3), "m3 m-3")

**📌 Takeaways**

- A Series is a labelled 1D array; a DataFrame is named columns sharing an index, each with its own dtype.
- Select with `.loc` (labels, slices include both ends), `.iloc` (positions), and boolean masks combined with `&` and `|`.
- Assign to a column, or to a masked `.loc` selection, to derive or update values without a loop.
- `join` combines on the index, `merge` on a shared column, and `concat` stacks; cells missing from one table come out `NaN`.
- `.plot()` wraps matplotlib for a quick line, bar, scatter, box, or histogram straight from a Series or DataFrame.
- Look at a file before reading it, then tell `read_csv` what it holds: `sep` for the separator, `na_values` for missing-value codes, `parse_dates` for time columns, `dtype` for keys.
- A datetime index enables date selection with `.loc`, `resample`, and `rolling`; xarray makes the same label/position split along named dimensions with `.sel` and `.isel`.
- `groupby` is split-apply-combine.
- Missing is not zero: detect with `isna`, rely on NaN-skipping reductions, and fill with `ffill`/`interpolate` only as a deliberate physical choice.
- Filling gaps with `fillna(0)` silently biases statistics whenever zero is a valid value.

## Summary

| Concept | Rule to remember |
|---|---|
| Series and DataFrame | A Series is a labelled 1D array; a DataFrame is named columns sharing one index. |
| Selecting | `.loc` by label (slices include both ends), `.iloc` by position, boolean masks to filter. |
| Deriving and modifying | Assign to a column, or to a masked `.loc` selection — no loop needed. |
| Combining | `join` on the index, `merge` on a column, `concat` to stack. |
| Reading | Look first, then be explicit: `sep`, `na_values`, `parse_dates`, `dtype`. |
| Time | A datetime index is what enables date slicing, `resample`, and `rolling`. |
| Selecting in xarray | `.sel` by label, `.isel` by position, each along a named dimension. |
| Grouping | `groupby` is split-apply-combine. |
| Missing data | Missing is not zero: detect with `isna`, and fill only as a deliberate physical choice. |
| The zero trap | `fillna(0)` silently biases every statistic whenever zero is itself a valid value. |

## Resources

- [Python for Data Analysis, 3rd ed. — Getting Started with pandas](https://wesmckinney.com/book/pandas-basics) — McKinney; Series/DataFrame mechanics, selection, and the data-cleaning chapter that follows it.
- [pandas — Getting started](https://pandas.pydata.org/docs/getting_started/index.html) — the official task-oriented tutorials for reading, selecting, grouping, and combining data.
- [Pandas: Groupby](https://earth-env-data-science.github.io/lectures/pandas/pandas_groupby.html) — from *Earth and Environmental Data Science*; `groupby` on real environmental data, beyond what this subchapter covers.